# Apply deterministic rules to PRESTAZIONI dataset

In [1]:
import pandas as pd
import re

### Define the correction function

In [2]:
def apply_corrections(text, substitution_dict, words_to_delete):
    words = text.lower().split()
    corrected_words = []
    for word in words:
        if word in words_to_delete:
            continue
        corrected_word = substitution_dict.get(word, word)
        corrected_words.append(corrected_word)
    return " ".join(corrected_words)


### Load datasets

In [3]:
# Load the nursing text dataset (tab-separated)
df = pd.read_csv("dataset/df_prestazioni_inf_clean_update.csv", sep="\t")

# Load the correction rules (semicolon-separated)
rules_df = pd.read_csv("vocabulary/vocab_FNOPI_corretto_istat.csv", sep=";")

In [4]:
df

,index,macroprestazione,testo,testo_pulito,cod_valoremedio_liquidazione,descr_cod_valoremedio_liquidazione
0,0,"Accesso Vascolare, Terapia Infusiva, Prelievo ...",Prelievi ematici,prelievi ematici,033T,Prelievo capillare e venoso del sangue o racco...
1,1,"Accesso Vascolare, Terapia Infusiva, Prelievo ...",Prelievi venosi,prelievi venosi,033T,Prelievo capillare e venoso del sangue o racco...
2,2,"Accesso Vascolare, Terapia Infusiva, Prelievo ...",Prelievi,prelievi,033T,Prelievo capillare e venoso del sangue o racco...
3,3,Medicazioni Avanzate,Medicazioni varie,medicazioni varie,031T,Medicazione semplice
4,4,procedure clinico assistenziali,Clistere /svuotamento manuale,clistere svuotamento manuale,080T,Preparazione ed effettuazione di clistere
...,...,...,...,...,...,...
22609,39482,"Accesso Vascolare, Terapia Infusiva, Prelievo ...",SOMMINISTRAZIONE TERAPIA ENDOVENOSA/INTRAMUSCO...,somministrazione terapia endovenosa intramusco...,015T,Somministrazione dei medicinali prescritti per...
22610,39483,Educazione Sanitaria,Addestramento del caregivers per la necessità ...,addestramento del caregivers per la necessità ...,122R,Accoglienza del paziente: presentazione di luo...
22611,39485,Educazione Sanitaria,Educazione e gestione pazienti con PEG,educazione e gestione pazienti con peg,055T,Preparazione e somministrazione di alimenti sp...
22612,39486,procedure clinico assistenziali,Sostituzione catetere vescicale,sostituzione catetere vescicale,065T,Assistenza ordinaria ad un paziente portatore ...


### Prepare substitution dictionary and deletion list

In [5]:
# Normalize relevant columns to lowercase
rules_df["word"] = rules_df["word"].str.lower()
rules_df["word_corretta"] = rules_df["word_corretta"].str.lower()

# Create substitution dictionary (exclude rows where 'word corretta' is 'eliminare')
substitutions_df = rules_df[rules_df["word_corretta"] != "eliminare"]
substitution_dict = dict(zip(substitutions_df["word"], substitutions_df["word_corretta"]))

# List of words to delete
words_to_delete = rules_df[rules_df["word_corretta"] == "eliminare"]["word"].tolist()


### Apply corrections to the dataset

In [6]:
# Apply the correction function to the 'testo_pulito' column
df["testo_corretto"] = df["testo_pulito"].apply(
    lambda x: apply_corrections(x, substitution_dict, words_to_delete)
)


# Preview the result
df[["testo_pulito", "testo_corretto"]].head(50)


,testo_pulito,testo_corretto
0,prelievi ematici,prelievo ematico
1,prelievi venosi,prelievo venoso
2,prelievi,prelievo
3,medicazioni varie,medicazione varie
4,clistere svuotamento manuale,clistere svuotamento manuale
5,prelievi ematici,prelievo ematico
6,medicazioni,medicazione
7,medicazioni,medicazione
8,educazione al caregiver,educazione al caregiver
9,medicazioni,medicazione


In [8]:
df.columns

Index(['index', 'macroprestazione', 'testo', 'testo_pulito',
       'cod_valoremedio_liquidazione', 'descr_cod_valoremedio_liquidazione',
       'testo_corretto'],
      dtype='object')

In [9]:
df = df[['index', 'macroprestazione', 'testo', 'testo_pulito', 'testo_corretto', 'cod_valoremedio_liquidazione', 'descr_cod_valoremedio_liquidazione']]

# Save the cleaned dataset (optional)
df.to_csv("dataset/df_prestazioni_inf_corretto.csv", sep="\t", index=False)